# The HWP fast axis offset, from a polarized standard

A calibration run, not a science one. It reduces a sequence on a star whose
polarization angle on sky is already known, asks what `theta_off` makes the
pipeline reproduce that angle, and reports the answer. What you do with it is
hand it to the science reduction:

```toml
fast_axis_method = "fixed"
theta_off = <the number this prints>
```

which is why none of it is wired into `polmode.run`: the standard is a
different target from the science target, usually a different night, and the
offset is the only thing that travels between them.

The other route to the same number is
`nirc2pol.polarimetry.fit_fast_axis_butterfly`, which reads it off the
orientation of a tangentially polarized disk. Neither checks the other in the
sense of sharing assumptions — the butterfly needs a disk and assumes it is
azimuthally polarized, this needs a catalogue and assumes that catalogue —
which is exactly what makes agreement between them worth something.

**One thing this exists to make hard to get wrong: fit it both ways.**
Leaving the leakage in displaces the offset; fitting it removes the
displacement and amplifies the noise. Which wins depends on the rotation span
and the per-cycle scatter, and the only reliable way to find out is to look at
both — so this prints both.

In [ ]:
import logging
import os

import numpy as np

from nirc2pol.polarimetry import (PolarizedStandard, curve_of_growth_polarization,
                                  fit_theta_off_polstd, measure_cycles,
                                  prepare_cycles)
from nirc2pol.polmode import run
from nirc2pol.reduction.config import ReductionConfig

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# The config for the standard-star night.
CONFIG_PATH = "reduction_config.toml"

## 1. What is known about the star

From outside this pipeline. **There is no bundled catalogue on purpose** — a
value written here can be traced to whoever chose it, and one shipped in the
package cannot.

If the catalogue value is optical, carry it to the observing band with
`nirc2pol.polarimetry.serkowski_p` first, and put an honest `p_err` on the
result: at L′ that extrapolation is worth a factor rather than a percent.

`theta_err` is the one that matters. It is what the fit weights by, and **the
catalogue angle propagates into `theta_off` at half its own error** — 2° here
is 1° on the answer. `p` and `theta` are not interchangeable inputs: `p` sets
the modulus of the fitted coefficient and so lands in the efficiency, while
`theta` sets its phase and so lands in `theta_off`. A badly extrapolated `p`
leaves `theta_off` almost untouched.

In [ ]:
STANDARD = PolarizedStandard(
    name="REPLACE ME",
    p=0.0093,               # fraction, not percent, in the observed band
    theta=103.0,            # sky position angle [deg]
    p_err=0.004,
    theta_err=2.0,
    band="Lp",
    reference="fill this in -- it is the one input that cannot be recomputed",
)

# The aperture. Read the radius off the curve of growth below rather than
# choosing it here: p is a ratio over a shared aperture, so PSF clipping
# cannot bias it, and a p that moves with radius is telling you about the
# background instead.
RADIUS = 30.0
BACKGROUND = (110.0, 165.0)     # annulus for the residual background plane
MASK = None                     # exclude a companion or a registration wedge

## 2. Reduce the night

`run` does the standard reduction and hands back what it built. `prepare_cycles`
then takes the matched cycles to the point where `theta_off` is still free, so
trying a value is one rotation rather than a re-reduction.

`register_method` comes from the config so the photometry sees the same
registration the products did. Nothing else to choose: `measure_cycles` uses
only the instrument-to-sky frame rotation, never the north angle, so the
north-up question does not arise for an aperture sum.

In [ ]:
cfg = ReductionConfig.from_toml(CONFIG_PATH)
products = run(cfg, config_path=CONFIG_PATH)
instrument, cycles = products["instrument"], products["cycles"]

prepared = prepare_cycles(instrument, cycles,
                          register_method=cfg.register_method)
print(f"{len(prepared)} complete HWP cycles")

## 3. Is the measurement stable?

The standing check on the background treatment, and the reason to run it
before reading any fit.

**Both columns should be flat.** Aperture losses cannot bias `p` — `q`, `u`
and `I` are integrated over the same pixels, so clipping the PSF divides out.
So if `p` climbs with radius or the angle walks, that is not the source: it is
the background, and no fit below is worth reading until it is fixed.

The usual culprit is a residual in **Q and U**, not in I. They are
differences, so the sky is expected to cancel — and it does not, quite. On the
2025-12-06 standard, U carried a detector-scale gradient worth +22 ADU/px at
the star: nothing beside a 1.2e5 core, but an annulus at r=60–80 holds ~9000
pixels and sums it into more signal than the star has out there. Left in, `p`
ran from 0.9% to 4.1% across this table.

In [ ]:
cog = curve_of_growth_polarization(prepared, cfg.theta_off,
                                   radii=[10, 20, 30, 40, 60, 80],
                                   background=BACKGROUND, mask=MASK)
print("curve of growth in polarization (theta_off held at the config value)")
print(f"  {'r [px]':>7} {'p [%]':>9} {'theta [deg]':>12}")
for r, p, t in zip(cog["radius"], cog["p"], cog["theta"]):
    print(f"  {r:7.0f} {100 * p:9.3f} {t:12.2f}")

## 4. How noisy is one cycle?

This decides whether the joint IP fit can work at all. The offset and the
leakage separate through field rotation, and the separation is amplified by
the condition number — so a large per-cycle scatter at a short rotation span
means the joint fit returns noise however well posed it looks.

In [ ]:
measured = measure_cycles(prepared, radius=RADIUS, background=BACKGROUND,
                          mask=MASK)
sky = measured.z * np.exp(-1j * np.radians(measured.base
                                           + 4.0 * cfg.theta_off))
sigma = float(np.hypot(sky.real.std(), sky.imag.std()) / np.sqrt(2))

print(f"aperture r = {RADIUS:.0f} px at ({measured.center[0]:.2f}, "
      f"{measured.center[1]:.2f})")
print(f"  measured p = {100 * abs(sky.mean()):.3f}%, "
      f"sky angle = {np.degrees(0.5 * np.angle(sky.mean())) % 180:.2f} deg "
      f"at theta_off = {cfg.theta_off}")
print(f"  per-cycle scatter on q, u = {100 * sigma:.3f}%")

## 5. The fit, both ways

`fit_ip=False` is the well-posed primitive and absorbs the instrumental
leakage into the answer. `fit_ip=True` solves for `ipq`, `ipu`, the efficiency
and `theta_off` together, in one closed-form complex least squares — no
optimizer — but it needs the field to have rotated enough to tell the leakage
(fixed in the instrument frame) from the source (fixed on sky).

In [ ]:
plain = fit_theta_off_polstd(prepared, STANDARD, radius=RADIUS,
                             background=BACKGROUND, mask=MASK)
joint = fit_theta_off_polstd(prepared, STANDARD, radius=RADIUS,
                             background=BACKGROUND, mask=MASK, fit_ip=True)

print(f"{'':14} {'theta_off':>18} {'efficiency':>12}")
print(f"  {'leakage left in':12} {plain.theta_off:+10.3f} "
      f"+/-{plain.theta_off_err:5.3f} {plain.efficiency:12.3f}")
print(f"  {'leakage fitted':12} {joint.theta_off:+10.3f} "
      f"+/-{joint.theta_off_err:5.3f} {joint.efficiency:12.3f}"
      f"   ipq {joint.ip.ipq:+.4f}  ipu {joint.ip.ipu:+.4f}")
print(f"  {joint.rotation_span:.1f} deg of field rotation, "
      f"condition number {joint.condition_number:.0f}")

difference = abs(plain.theta_off - joint.theta_off)

# Efficiency is a fraction of the true polarization the system recovers, so
# it cannot exceed 1. A joint fit that reports more than that has not
# measured the leakage -- it has absorbed the source into it -- and the
# difference from the plain fit is then not evidence of anything.
unphysical = not (0.0 < joint.efficiency <= 1.05)
if unphysical:
    print(f"\n  The joint fit returns efficiency {joint.efficiency:.3f}, "
          f"which is unphysical -- no system recovers more than all of the\n"
          f"  polarization. At {joint.rotation_span:.1f} deg of rotation and "
          f"condition number {joint.condition_number:.0f} it cannot separate\n"
          f"  the leakage from the source, so it has fitted one as the other. "
          f"Use the FIRST number and state that it carries\n  the leakage. "
          f"To do better, pool a second standard at a different sky angle.")
elif difference < joint.theta_off_err:
    print(f"\n  The two differ by {difference:.2f} deg against a joint-fit "
          f"error of {joint.theta_off_err:.2f} deg, so the leakage is NOT "
          f"being measured here.\n  Use the first number and state that it "
          f"carries the leakage -- on synthetics that displaces theta_off by "
          f"of order 0.7 deg.\n  To do better, observe a second standard at a "
          f"different sky angle and pool them:\n"
          f"      fit_theta_off_polstd([(cycles_a, std_a), (cycles_b, std_b)], "
          f"fit_ip=True)")
else:
    print(f"\n  The two differ by {difference:.2f} deg, more than the joint "
          f"fit's {joint.theta_off_err:.2f} deg error, so the leakage is "
          f"real and being removed.\n  Use the second number.")

## 6. Report the dependence, not just the number

The catalogue angle is the input least likely to be right, and the whole
dependence on it is one line. Print the line rather than only the point
estimate, and the next person can re-evaluate it when a better value turns up
without reducing the night again.

In [ ]:
reference_angle = float(np.degrees(0.5 * np.angle(sky.mean())) % 180.0)
print("For any other catalogue angle, without re-reducing:")
print(f"    theta_off = {cfg.theta_off} - (theta_known - "
      f"{reference_angle:.2f}) / 2")